# Sparse Connectivity in V1 Networks

Explore the structure of large-scale sparse synaptic connectivity.

## Table of Contents
1. [Why Sparse Matrices?](#1-why-sparse-matrices)
2. [COO vs BCOO Format](#2-coo-vs-bcoo-format)
3. [Loading Network Data](#3-loading-network-data)
4. [Connection Statistics](#4-connection-statistics)
5. [Delay Distribution](#5-delay-distribution)
6. [Connectivity Visualization](#6-connectivity-visualization)
7. [Dale's Law Constraints](#7-dales-law-constraints)

---

In [ ]:
# Setup
import jax
import jax.numpy as jnp
from jax.experimental import sparse as jsparse
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Enable high-DPI display for plots
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

In [ ]:
# Configuration - MODIFY THIS PATH
DATA_DIR = Path("/path/to/GLIF_network")  # <-- CHANGE THIS

# Verify data exists
required_files = [
    DATA_DIR / "network_dat.pkl",
    DATA_DIR / "network" / "v1_nodes.h5",
]

all_found = True
for f in required_files:
    if f.exists():
        print(f"Found: {f.name}")
    else:
        print(f"MISSING: {f}")
        all_found = False

if not all_found:
    print("\nPlease update DATA_DIR to point to your GLIF_network directory.")

## 1. Why Sparse Matrices?

The V1 model has **~51,000 neurons**. A dense connectivity matrix would require:

$$51,316 \times 51,316 \approx 2.6 \text{ billion parameters}$$

At 32-bit precision: $2.6B \times 4 = \mathbf{10.4 GB}$ just for one weight matrix!

However, biological networks are **sparse**. Cortical connectivity is typically ~10-15% dense.

### Memory Comparison

| Format | Storage for V1 Network |
|--------|------------------------|
| Dense | ~10 GB |
| COO (10% density) | ~1 GB |
| BCOO (compressed) | ~0.5 GB |

In [ ]:
# Demonstrate sparse vs dense memory
n_neurons = 51316
density = 0.1  # 10% connectivity

# Dense matrix size
dense_elements = n_neurons * n_neurons
dense_bytes = dense_elements * 4  # float32

# Sparse COO size (indices + values)
n_connections = int(dense_elements * density)
# Each connection: 2 indices (int64) + 1 weight (float32)
sparse_bytes = n_connections * (2 * 8 + 4)

print(f"V1 Network: {n_neurons:,} neurons")
print(f"Connectivity density: {density*100:.0f}%")
print(f"")
print(f"Dense matrix:")
print(f"  Elements: {dense_elements:,}")
print(f"  Memory: {dense_bytes / 1e9:.1f} GB")
print(f"")
print(f"Sparse COO format:")
print(f"  Non-zero elements: {n_connections:,}")
print(f"  Memory: {sparse_bytes / 1e9:.2f} GB")
print(f"")
print(f"Memory savings: {(1 - sparse_bytes/dense_bytes)*100:.1f}%")

## 2. COO vs BCOO Format

### COO (Coordinate) Format
- Stores `(row, col, value)` for each non-zero element
- Simple and flexible
- Good for construction, less efficient for operations

### BCOO (Batched COO) - JAX's Format
- JAX's sparse matrix format in `jax.experimental.sparse`
- Stores `(indices, data)` where indices is `(nnz, 2)`
- JIT-compilable and GPU-compatible
- Supports batching for multiple matrices

In [ ]:
# Demonstrate BCOO format in JAX
from jax.experimental.sparse import BCOO

# Create a small example sparse matrix
# 5x5 matrix with 7 non-zero elements
dense_matrix = jnp.array([
    [1.0, 0.0, 2.0, 0.0, 0.0],
    [0.0, 3.0, 0.0, 4.0, 0.0],
    [5.0, 0.0, 0.0, 0.0, 6.0],
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 7.0, 0.0, 0.0, 0.0],
])

# Convert to BCOO
sparse_matrix = BCOO.fromdense(dense_matrix)

print("Dense matrix:")
print(dense_matrix)
print(f"\nBCOO representation:")
print(f"  Indices shape: {sparse_matrix.indices.shape}")
print(f"  Data shape: {sparse_matrix.data.shape}")
print(f"  Indices (row, col):")
print(sparse_matrix.indices)
print(f"  Values:")
print(sparse_matrix.data)

In [ ]:
# Sparse matrix-vector multiplication
x = jnp.ones(5)

# Dense matmul
dense_result = dense_matrix @ x

# Sparse matmul
sparse_result = sparse_matrix @ x

print(f"Input vector: {x}")
print(f"Dense result: {dense_result}")
print(f"Sparse result: {sparse_result}")
print(f"Match: {jnp.allclose(dense_result, sparse_result)}")

In [ ]:
# JIT compilation of sparse operations
@jax.jit
def sparse_forward(sparse_mat, x):
    return sparse_mat @ x

# Compile and run
result = sparse_forward(sparse_matrix, x)
print(f"JIT-compiled sparse result: {result}")

# Timing comparison
import time

# Create larger matrices for timing
n = 1000
density = 0.1
key = jax.random.PRNGKey(42)

# Random sparse matrix
k1, k2 = jax.random.split(key)
mask = jax.random.uniform(k1, (n, n)) < density
values = jax.random.normal(k2, (n, n)) * mask
dense_large = values
sparse_large = BCOO.fromdense(dense_large)
x_large = jax.random.normal(key, (n,))

# JIT compile
_ = jax.jit(lambda m, x: m @ x)(dense_large, x_large)
_ = jax.jit(lambda m, x: m @ x)(sparse_large, x_large)

print(f"\nMatrix size: {n}x{n}, density: {density*100:.0f}%")
print(f"Non-zeros: {int(n*n*density):,}")

## 3. Loading Network Data

The Billeh network data contains:
- Neuron positions (x, y, z)
- Node parameters (V_th, g, E_L, etc.)
- Synapse indices, weights, and delays

In [ ]:
try:
    from v1_jax.data import load_billeh
    
    # Load a subset of the network for exploration
    # Use n_neurons=None for full network
    n_neurons_to_load = 10000  # Subset for faster loading
    
    print(f"Loading Billeh network ({n_neurons_to_load:,} neurons)...")
    input_pop, network, bkg_weights = load_billeh(
        n_input=17400,
        n_neurons=n_neurons_to_load,
        core_only=True,
        data_dir=str(DATA_DIR),
        seed=3000,
    )
    
    print(f"\nNetwork loaded successfully!")
    print(f"  Neurons: {network['n_nodes']:,}")
    print(f"  Synapses: {network['n_edges']:,}")
    
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please update DATA_DIR to point to your GLIF_network directory.")
    network = None

In [ ]:
if network is not None:
    # Explore network structure
    print("Network dict keys:")
    for key, value in network.items():
        if isinstance(value, np.ndarray):
            print(f"  {key}: array of shape {value.shape}")
        elif isinstance(value, dict):
            print(f"  {key}: dict with {len(value)} items")
        else:
            print(f"  {key}: {type(value).__name__}")
    
    print("\nSynapses dict keys:")
    for key, value in network['synapses'].items():
        if isinstance(value, np.ndarray):
            print(f"  {key}: array of shape {value.shape}, dtype={value.dtype}")
        else:
            print(f"  {key}: {value}")

## 4. Connection Statistics

Let's analyze the connectivity structure: excitatory vs inhibitory, in/out degree distributions.

In [ ]:
if network is not None:
    indices = network['synapses']['indices']
    weights = network['synapses']['weights']
    delays = network['synapses']['delays']
    n_neurons = network['n_nodes']
    
    print("Connectivity Statistics")
    print("=" * 50)
    
    # Sparsity
    max_synapses = n_neurons * n_neurons
    actual_synapses = len(weights)
    density = actual_synapses / max_synapses
    sparsity = 1 - density
    
    print(f"\nDensity and Sparsity:")
    print(f"  Maximum possible synapses: {max_synapses:,}")
    print(f"  Actual synapses: {actual_synapses:,}")
    print(f"  Density: {density*100:.2f}%")
    print(f"  Sparsity: {sparsity*100:.2f}%")
    
    # E/I statistics
    n_exc = np.sum(weights > 0)
    n_inh = np.sum(weights < 0)
    
    print(f"\nExcitatory/Inhibitory Balance:")
    print(f"  Excitatory synapses: {n_exc:,} ({100*n_exc/len(weights):.1f}%)")
    print(f"  Inhibitory synapses: {n_inh:,} ({100*n_inh/len(weights):.1f}%)")
    print(f"  E/I ratio: {n_exc/n_inh:.2f}")
    
    # Weight statistics
    print(f"\nWeight Statistics:")
    print(f"  Min: {weights.min():.4f}")
    print(f"  Max: {weights.max():.4f}")
    print(f"  Mean: {weights.mean():.4f}")
    print(f"  Std: {weights.std():.4f}")
    print(f"  Mean (E only): {weights[weights > 0].mean():.4f}")
    print(f"  Mean (I only): {weights[weights < 0].mean():.4f}")

In [ ]:
if network is not None:
    # Degree distributions
    # Note: indices[:, 0] are post-synaptic (with receptor multiplexing)
    # indices[:, 1] are pre-synaptic
    
    # Remove receptor multiplexing from post indices
    post_indices = indices[:, 0] // 4  # Divide by 4 to get neuron index
    pre_indices = indices[:, 1]
    
    in_degree = np.bincount(post_indices, minlength=n_neurons)
    out_degree = np.bincount(pre_indices, minlength=n_neurons)
    
    print("\nDegree Statistics:")
    print(f"  In-degree:  mean={in_degree.mean():.1f}, std={in_degree.std():.1f}, max={in_degree.max()}")
    print(f"  Out-degree: mean={out_degree.mean():.1f}, std={out_degree.std():.1f}, max={out_degree.max()}")
    
    # Plot degree distributions
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(in_degree, bins=50, alpha=0.7, edgecolor='black')
    axes[0].axvline(x=in_degree.mean(), color='red', linestyle='--', label=f'Mean: {in_degree.mean():.0f}')
    axes[0].set_xlabel('In-degree')
    axes[0].set_ylabel('Count')
    axes[0].set_title('In-Degree Distribution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(out_degree, bins=50, alpha=0.7, edgecolor='black')
    axes[1].axvline(x=out_degree.mean(), color='red', linestyle='--', label=f'Mean: {out_degree.mean():.0f}')
    axes[1].set_xlabel('Out-degree')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Out-Degree Distribution')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if network is not None:
    # Weight distributions
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    # All weights
    axes[0].hist(weights, bins=100, alpha=0.7, edgecolor='black')
    axes[0].axvline(x=0, color='gray', linestyle='--')
    axes[0].set_xlabel('Weight')
    axes[0].set_ylabel('Count')
    axes[0].set_title('All Weights Distribution')
    axes[0].grid(True, alpha=0.3)
    
    # Excitatory weights
    exc_weights = weights[weights > 0]
    axes[1].hist(exc_weights, bins=50, alpha=0.7, color='red', edgecolor='black')
    axes[1].set_xlabel('Weight')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Excitatory Weights (n={len(exc_weights):,})')
    axes[1].grid(True, alpha=0.3)
    
    # Inhibitory weights
    inh_weights = weights[weights < 0]
    axes[2].hist(inh_weights, bins=50, alpha=0.7, color='blue', edgecolor='black')
    axes[2].set_xlabel('Weight')
    axes[2].set_ylabel('Count')
    axes[2].set_title(f'Inhibitory Weights (n={len(inh_weights):,})')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Delay Distribution

Synaptic delays model the time for signal transmission between neurons. They depend on:
- Axonal conduction velocity
- Distance between neurons
- Synapse type

In [ ]:
if network is not None:
    print("Delay Statistics")
    print("=" * 50)
    print(f"Min delay: {delays.min():.1f} ms")
    print(f"Max delay: {delays.max():.1f} ms")
    print(f"Mean delay: {delays.mean():.1f} ms")
    print(f"Median delay: {np.median(delays):.1f} ms")
    
    # Unique delay values
    unique_delays = np.unique(delays)
    print(f"\nUnique delay values: {len(unique_delays)}")
    if len(unique_delays) <= 10:
        print(f"Delays: {unique_delays}")
    
    # Plot delay distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(delays, bins=30, alpha=0.7, edgecolor='black')
    axes[0].axvline(x=delays.mean(), color='red', linestyle='--', label=f'Mean: {delays.mean():.1f}ms')
    axes[0].set_xlabel('Synaptic Delay (ms)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Synaptic Delay Distribution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Delay vs weight
    sample_idx = np.random.choice(len(delays), min(5000, len(delays)), replace=False)
    axes[1].scatter(delays[sample_idx], weights[sample_idx], s=1, alpha=0.3)
    axes[1].axhline(y=0, color='gray', linestyle='--')
    axes[1].set_xlabel('Delay (ms)')
    axes[1].set_ylabel('Weight')
    axes[1].set_title('Delay vs Weight (sampled)')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. Connectivity Visualization

Visualize the sparse connectivity matrix structure.

In [ ]:
if network is not None:
    # Visualize sampled connectivity matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Sample connections for visualization
    key = jax.random.PRNGKey(42)
    sample_size = min(10000, len(indices))
    sample_idx = jax.random.choice(key, len(indices), (sample_size,), replace=False)
    sample_idx = np.array(sample_idx)
    
    # Remove receptor multiplexing
    sampled_post = indices[sample_idx, 0] // 4
    sampled_pre = indices[sample_idx, 1]
    sampled_weights = weights[sample_idx]
    
    # Scatter plot of connectivity
    scatter = axes[0].scatter(
        sampled_pre,
        sampled_post,
        c=sampled_weights,
        cmap='RdBu_r',
        s=1,
        alpha=0.5,
        vmin=-np.abs(sampled_weights).max(),
        vmax=np.abs(sampled_weights).max(),
    )
    plt.colorbar(scatter, ax=axes[0], label='Weight')
    axes[0].set_xlabel('Presynaptic Neuron')
    axes[0].set_ylabel('Postsynaptic Neuron')
    axes[0].set_title(f'Sampled Connectivity Matrix ({sample_size:,} synapses)')
    
    # Zoomed view on a subset
    subset_size = 500
    in_range = (sampled_pre < subset_size) & (sampled_post < subset_size)
    
    scatter2 = axes[1].scatter(
        sampled_pre[in_range],
        sampled_post[in_range],
        c=sampled_weights[in_range],
        cmap='RdBu_r',
        s=10,
        alpha=0.7,
        vmin=-np.abs(sampled_weights[in_range]).max() if np.any(in_range) else -1,
        vmax=np.abs(sampled_weights[in_range]).max() if np.any(in_range) else 1,
    )
    plt.colorbar(scatter2, ax=axes[1], label='Weight')
    axes[1].set_xlabel('Presynaptic Neuron')
    axes[1].set_ylabel('Postsynaptic Neuron')
    axes[1].set_title(f'Zoomed View (first {subset_size} neurons)')
    
    plt.tight_layout()
    plt.show()

In [ ]:
if network is not None:
    # Spatial connectivity: distance vs connection probability
    x = network['x']
    y = network['y']
    z = network['z']
    
    # Sample connections and compute distances
    sample_size = min(50000, len(indices))
    sample_idx = np.random.choice(len(indices), sample_size, replace=False)
    
    post_neurons = indices[sample_idx, 0] // 4
    pre_neurons = indices[sample_idx, 1]
    
    # Euclidean distance between connected neurons
    dx = x[post_neurons] - x[pre_neurons]
    dy = y[post_neurons] - y[pre_neurons]
    dz = z[post_neurons] - z[pre_neurons]
    distances = np.sqrt(dx**2 + dy**2 + dz**2)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Distance distribution
    axes[0].hist(distances, bins=50, alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Distance (um)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Distance Between Connected Neurons')
    axes[0].grid(True, alpha=0.3)
    
    # Weight vs distance
    axes[1].scatter(distances, weights[sample_idx], s=1, alpha=0.3)
    axes[1].axhline(y=0, color='gray', linestyle='--')
    axes[1].set_xlabel('Distance (um)')
    axes[1].set_ylabel('Weight')
    axes[1].set_title('Weight vs Distance')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Mean connection distance: {distances.mean():.1f} um")
    print(f"Max connection distance: {distances.max():.1f} um")

## 7. Dale's Law Constraints

**Dale's Law**: A neuron is either excitatory OR inhibitory, never both.

- Excitatory neurons: All outgoing weights are **positive**
- Inhibitory neurons: All outgoing weights are **negative**

This biological constraint is enforced during training to maintain biological plausibility.

In [ ]:
from v1_jax.nn.constraints import apply_dale_constraint, dale_law_projection

# Demonstrate Dale's law constraint
print("Dale's Law Constraint Demo")
print("=" * 50)

# Example weights (some violate Dale's law)
test_weights = jnp.array([0.5, -0.3, 0.2, -0.1, 0.4, -0.2])
is_excitatory = jnp.array([True, True, True, False, False, False])

print("\nOriginal weights:")
print(f"  Weights:        {test_weights}")
print(f"  Is excitatory:  {is_excitatory}")

# Apply Dale's constraint
constrained = apply_dale_constraint(test_weights, is_excitatory)

print("\nAfter Dale's Law constraint:")
print(f"  Constrained:    {constrained}")

print("\nExplanation:")
print("  - Excitatory neurons: negative weights become 0")
print("  - Inhibitory neurons: positive weights become 0")

In [ ]:
if network is not None:
    # Verify Dale's law in loaded network
    print("Verifying Dale's Law in Network")
    print("=" * 50)
    
    # For each presynaptic neuron, check if all weights have same sign
    pre_neurons = indices[:, 1]
    
    violations = 0
    checked = 0
    
    for neuron_id in range(min(1000, n_neurons)):  # Check first 1000 neurons
        mask = pre_neurons == neuron_id
        if np.sum(mask) > 1:  # Need at least 2 connections to check
            checked += 1
            neuron_weights = weights[mask]
            
            # Check if all same sign
            all_positive = np.all(neuron_weights > 0)
            all_negative = np.all(neuron_weights < 0)
            
            if not (all_positive or all_negative):
                violations += 1
    
    print(f"Neurons checked: {checked}")
    print(f"Dale's law violations: {violations}")
    
    if violations == 0:
        print("\nNetwork satisfies Dale's law!")
    else:
        print(f"\n{violations} neurons have mixed E/I weights (may need constraint enforcement)")

In [ ]:
# Demonstrate JIT-compatible Dale constraint
@jax.jit
def forward_with_dale(weights, is_exc, inputs):
    """Apply Dale's law before forward pass."""
    constrained_weights = apply_dale_constraint(weights, is_exc)
    return constrained_weights @ inputs

# Test
n = 10
weights_test = jax.random.normal(jax.random.PRNGKey(0), (n, n))
is_exc_test = jax.random.uniform(jax.random.PRNGKey(1), (n,)) > 0.5
inputs_test = jax.random.normal(jax.random.PRNGKey(2), (n,))

output = forward_with_dale(weights_test, is_exc_test, inputs_test)
print(f"JIT-compiled forward pass with Dale's law: shape={output.shape}")

---

## Summary

Key takeaways:

1. **Sparse Efficiency**: Biological networks are sparse (~10% density), saving 90%+ memory

2. **BCOO Format**: JAX's sparse format is JIT-compatible and GPU-friendly

3. **Network Structure**:
   - V1 has ~51K neurons with ~300M synapses
   - Mixture of excitatory (~80%) and inhibitory (~20%) neurons
   - Connectivity depends on spatial distance

4. **Synaptic Delays**: Range from ~1-5ms depending on distance and synapse type

5. **Dale's Law**: Neurons are either E or I, never both; enforced as weight constraint

---

## Exercises

1. **Calculate sparsity**: Load the full 51K network and compute exact sparsity percentage.

2. **Connection probability vs distance**: Bin connections by distance and compute probability.

3. **Layer analysis**: If position data includes layer info, analyze connectivity between layers.

In [ ]:
# Exercise starter: Connection probability vs distance

# Distance bins
distance_bins = np.arange(0, 500, 50)  # 0-50, 50-100, etc.

print("TODO: Implement connection probability vs distance analysis")
print("Hint: For each distance bin, count connections and divide by possible connections")

---

## Source Files

- `src/v1_jax/nn/sparse_layer.py` - Sparse connectivity layers
- `src/v1_jax/nn/constraints.py` - Dale's law and weight constraints
- `src/v1_jax/data/network_loader.py` - Network loading utilities

## References

- Chen et al., "Data-driven models of visual cortex", Science Advances 2022
- Billeh et al., "Systematic Integration of Structural and Functional Data into Multi-scale Models of Mouse Primary Visual Cortex", Neuron 2020
- JAX Sparse Documentation: https://jax.readthedocs.io/en/latest/jax.experimental.sparse.html